In [16]:
from collections import defaultdict
import torch
import pandas as pd
import torch.nn.functional as F
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T
from torch_geometric.nn import SAGEConv, to_hetero, GATConv, GCNConv, GATv2Conv, HeteroConv
import torch.nn as nn
from torch_geometric.loader import LinkNeighborLoader, NeighborLoader
from torch.utils.data import DataLoader
from torch_geometric.sampler import NegativeSampling
from torch_geometric.datasets import AmazonBook, MovieLens100K, DBLP, IMDB

import pickle
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
with open("../data/dicts/bert_embeddings/protein_esm_35M.pkl", 'rb') as f:
    emb_prot = pickle.load(f)
with open("../data/dicts/bert_embeddings/sm_chemberta_10M_MTR.pkl", 'rb') as f:
    emb_sm = pickle.load(f)
with open("../data/dicts/bert_embeddings/rna_berta.pkl", 'rb') as f:
    emb_rna = pickle.load(f)
with open("../data/dicts/bert_embeddings/dna_full.pkl", 'rb') as f:
    emb_dna = pickle.load(f)

rawid2enb = emb_prot | emb_sm | emb_rna | emb_dna


In [2]:
node_types = ['AA', 'DNA', 'RNA', 'SmallMolecule']

rawid_to_local = {}
raw_tensors = {}

for t_name in node_types:
    # Читаем ТОЛЬКО колонку с ID, чтобы узнать тип узлов
    df_nodes = pd.read_csv(f'../data/nodes/nodes_for_mdm/{t_name}.csv', usecols=['id_entity'])

    embeddings_list = []

    for local_idx, raw_id in enumerate(df_nodes['id_entity']):
        # Заполняем маппинги для DataLoader
        rawid_to_local[raw_id] = local_idx

        # Достаем готовый эмбеддинг из твоего словаря
        #emb = rawid2enb[raw_id]

        #embeddings_list.append(emb)

    # Склеиваем список тензоров в одну матрицу для этого типа
    # Получится тензор размерности [N_nodes_of_this_type, embedding_dim]
    #raw_tensors[str(t_name)] = torch.stack(embeddings_list)

df_edges = pd.read_csv('../data/edges/clean_edges_without_NaNm.csv')


df_edges['id_entity_1'] = df_edges['id_entity_1'].apply(lambda x: rawid_to_local[x])
df_edges['id_entity_2'] = df_edges['id_entity_2'].apply(lambda x: rawid_to_local[x])

data = HeteroData()
# for k, v in raw_tensors.items():
#     data[k].x = v.float()

data['AA'].x = torch.randn((70937, 480))
data['DNA'].x = torch.randn((635, 768))
data['RNA'].x = torch.randn((738, 512))
data['SmallMolecule'].x = torch.randn((1001342, 384))

unique_pairs = df_edges[['type_entity_1', 'predicate', 'type_entity_2']].drop_duplicates()

for row in unique_pairs.itertuples(index=False):
    t1 = row.type_entity_1
    r = row.predicate
    t2 = row.type_entity_2

    # Фильтруем основной DataFrame по обоим условиям сразу
    filtered_df = df_edges[(df_edges['type_entity_1'] == t1) &
                           (df_edges['predicate'] == r) &
                           (df_edges['type_entity_2'] == t2)]

    arr = filtered_df[['id_entity_1', 'id_entity_2']].to_numpy().T
    if (t1 == 'AA') and (r == 'interacts_with') and (t2 == 'AA'):
        AA_arr = arr
        AA_rev_arr = AA_arr[[1, 0], :]
    else:
        data[t1, r, t2].edge_index = torch.tensor(arr, dtype=torch.long)

transform = T.ToUndirected(merge=True)
data = transform(data)
data['AA', 'interacts_with', 'AA'].edge_index = torch.tensor(AA_arr, dtype=torch.long)
data['AA', 'rev_interacts_with', 'AA'].edge_index = torch.tensor(AA_rev_arr, dtype=torch.long)

In [17]:
dataset = IMDB(root='../data/IMDB')
data = dataset[0]
data

Extracting ../data/IMDB/raw/IMDB_processed.zip
Processing...
Done!


HeteroData(
  movie={
    x=[4278, 3066],
    y=[4278],
    train_mask=[4278],
    val_mask=[4278],
    test_mask=[4278],
  },
  director={ x=[2081, 3066] },
  actor={ x=[5257, 3066] },
  (movie, to, director)={ edge_index=[2, 4278] },
  (movie, to, actor)={ edge_index=[2, 12828] },
  (director, to, movie)={ edge_index=[2, 4278] },
  (actor, to, movie)={ edge_index=[2, 12828] }
)

In [18]:
iw_edge_types = [i for i in data.edge_types if i[1] == 'to']

In [3]:
# iw_edge_types = [i for i in data.edge_types if i[1] == 'interacts_with']
# rev_iw_edge_types = [i for i in data.edge_types if i[1] == 'rev_interacts_with']
# iw_edge_types = [('AA', 'interacts_with', 'SmallMolecule')]
# rev_iw_edge_types = [('SmallMolecule', 'rev_interacts_with', 'AA')]


In [19]:
transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=False,
    neg_sampling_ratio=0,
    add_negative_train_samples=False,
    edge_types=iw_edge_types,
    #rev_edge_types=rev_iw_edge_types,
)

train_data, val_data, test_data = transform(data)

In [20]:
train_loaders = {}

for e_type in iw_edge_types:
    train_loaders[e_type] = LinkNeighborLoader(
        data=train_data,
        num_neighbors=[25, 10],  # Количество соседей для сэмплирования на 1 и 2 слоях
        edge_label_index=(e_type, train_data[e_type].edge_label_index),
        edge_label=train_data[e_type].edge_label,
        batch_size=1024,
        neg_sampling=NegativeSampling(mode="binary", amount=10),
        #neg_sampling_ratio=10,
        shuffle=True,
)

In [21]:
def train_margin(model, train_loaders, optimizer, margin=1.5):
    model.train()
    total_loss = 0
    num_pos_total = 0

    for edge_type, train_loader in train_loaders.items():
        for batch in tqdm(train_loader):
            batch = batch.to(device)
            optimizer.zero_grad()

            # 1. Получаем эмбеддинги один раз для всего батча
            z_dict = model.encoder(batch.x_dict, batch.edge_index_dict)

            # 2. Извлекаем индексы и определяем количество позитивных примеров
            edge_label_index = batch[edge_type].edge_label_index
            num_pos = (batch[edge_type].edge_label == 1).sum().item()
            amount = (batch[edge_type].edge_label == 0).sum().item() // num_pos

            # 3. Считаем скоры для позитивных ребер
            pos_idx = edge_label_index[:, :num_pos]
            pos_scores = model.decoder(z_dict, pos_idx, edge_type)

            # 4. Считаем скоры для всех негативных кандидатов
            neg_idx = edge_label_index[:, num_pos:]
            neg_scores_all = model.decoder(z_dict, neg_idx, edge_type)

            # 5. Hard Negative Mining:
            # Решейпим негативы в [num_pos, amount] и берем максимум по строке
            neg_scores_reshaped = neg_scores_all.view(num_pos, amount)
            hard_neg_scores, _ = torch.max(neg_scores_reshaped, dim=1)

            # 6. Loss: требуем, чтобы pos_scores > hard_neg_scores + margin
            # Используем MarginRankingLoss: loss = max(0, -y * (p1 - p2) + margin)
            # При y=1 он минимизирует max(0, margin - (pos - neg))
            target = torch.ones_like(pos_scores)
            loss = F.margin_ranking_loss(pos_scores, hard_neg_scores, target, margin=margin)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * num_pos
            num_pos_total += num_pos

    return total_loss / num_pos_total

def train_bce(model, train_loaders, optimizer):
    model.train()
    total_loss = 0
    total_examples = 0

    for edge_type, train_loader in train_loaders.items():
        for batch in tqdm(train_loader):
            batch = batch.to(device) # Убедитесь, что device определен в вашей области видимости
            optimizer.zero_grad()

            # 1. Получаем предсказания (логиты)
            # Важно: модель по-прежнему не должна использовать Sigmoid на выходе
            pred = model(batch.x_dict, batch.edge_index_dict,
                         batch[edge_type].edge_label_index, edge_type).squeeze()

            # 2. Подготовка целевых меток
            # Для BCE нужны вероятностные метки в диапазоне [0, 1]
            target = batch[edge_type].edge_label.float()

            # 3. Вычисление Binary Cross Entropy Loss
            # Используем with_logits для числовой стабильности (Sigmoid применяется внутри)
            loss = F.binary_cross_entropy_with_logits(pred, target)

            loss.backward()
            optimizer.step()

            total_loss += loss.detach().item() * pred.numel()
            total_examples += pred.numel()

    return total_loss / total_examples


class GNNEncoder(nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()

        # h(0): Одной проекции обычно достаточно
        self.lin0 = nn.LazyLinear(hidden_channels)

        # h(1) и h(2)
        self.conv1 = SAGEConv((-1, -1), hidden_channels, aggr = 'sum')
        self.norm1 = nn.LayerNorm(hidden_channels)

        self.conv2 = SAGEConv((-1, -1), hidden_channels, aggr = 'sum')
        self.norm2 = nn.LayerNorm(hidden_channels)

        # Финальный блок агрегации
        self.lin_out = nn.Sequential(
            nn.LazyLinear(2 * out_channels), # Промежуточное сжатие
            nn.ReLU(),
            nn.LayerNorm(2 * out_channels),
            nn.LazyLinear(out_channels)
        )

    def forward(self, x, edge_index):
        # h(0) - Начальные признаки
        h0 = self.lin0(x).relu()

        # h(1) - Первый порядок соседей
        h1 = self.conv1(h0, edge_index)
        h1 = self.norm1(h1).relu()

        # h(2) - Второй порядок соседей
        h2 = self.conv2(h1, edge_index)
        h2 = self.norm2(h2).relu()

        x_final = torch.cat([h0, h1, h2], dim=1)
        x = self.lin_out(x_final)
        return F.normalize(x, p=2, dim=-1)

class DotEdgeDecoder(torch.nn.Module):
    def forward(self, z_dict, edge_label_index, edge_type):
        row, col = edge_label_index

        # Извлекаем эмбеддинги для источника и цели
        z_src = z_dict[edge_type[0]][row]
        z_dst = z_dict[edge_type[-1]][col]

        # Вычисляем скалярное произведение (поэлементное умножение и сумма по последней оси)
        # Формат: (batch_size, hidden_channels) -> (batch_size,)
        z = (z_src * z_dst).sum(dim=-1)

        return z


class OldModel(torch.nn.Module):
    def __init__(self, hidden_channels, output_channels, data_metadata):
        super().__init__()
        self.encoder = GNNEncoder(hidden_channels, output_channels)
        # Передаем data_metadata в __init__, чтобы избежать обращения к глобальной переменной data
        self.encoder = to_hetero(self.encoder, data_metadata, aggr='sum')

        # Используем новый декодер
        self.decoder = DotEdgeDecoder()

    def forward(self, x_dict, edge_index_dict, edge_label_index, edge_type):
        z_dict = self.encoder(x_dict, edge_index_dict)
        return self.decoder(z_dict, edge_label_index, edge_type)

In [38]:
model = OldModel(hidden_channels=256, output_channels=64, data_metadata=data.metadata()).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for i in range(10):
    total_loss = train_bce(model, train_loaders, optimizer)
    print(total_loss)

100%|██████████| 11/11 [00:00<00:00, 20.63it/s]


0.443340219613894


100%|██████████| 11/11 [00:00<00:00, 20.70it/s]


0.405394638807435


100%|██████████| 11/11 [00:00<00:00, 20.10it/s]


0.40434434355036825


100%|██████████| 11/11 [00:00<00:00, 19.70it/s]


0.4034812363231482


 25%|██▌       | 1/4 [00:00<00:00,  9.14it/s]


KeyboardInterrupt: 

In [23]:
def get_embeddigs(model, test_data):
    all_embeddings = {node_type: [] for node_type in data.node_types}
    model.eval()
    # 4. Проход по батчам для каждого типа узлов
    with torch.no_grad():
        for node_type in data.node_types:

            # Создаем лоадер для конкретного типа узлов
            loader = NeighborLoader(
                data=test_data,
                # Указываем, сколько соседей сэмплировать на 1-м и 2-м слое (т.к. у нас 2 слоя SAGEConv)
                # Использование [-1] означает взятие всех соседей
                num_neighbors=[25, 10],
                input_nodes=node_type,
                batch_size=4096,
                shuffle=False, # Строго False, чтобы итоговые тензоры совпадали по порядку с исходными узлами
            )

            for batch in tqdm(loader):
                # batch - это подграф, содержащий целевые узлы и их соседей (в том числе других типов)
                batch = batch.to(device)
                out_dict = model.encoder(batch.x_dict, batch.edge_index_dict)

                # В PyG целевые узлы, для которых генерировался батч,
                # всегда находятся в самом начале тензора.
                # batch[node_type].batch_size хранит количество именно этих целевых узлов.
                batch_size = batch[node_type].batch_size

                # Срезаем только эмбеддинги целевых узлов, отбрасывая подтянутых соседей,
                # и переносим на CPU, чтобы не переполнить видеопамять (OOM)
                node_embeddings = out_dict[node_type][:batch_size].cpu()

                all_embeddings[node_type].append(node_embeddings)

    # 5. Конкатенация списков в единые тензоры
    for node_type in data.node_types:
        all_embeddings[node_type] = torch.cat(all_embeddings[node_type], dim=0)

    return all_embeddings

def get_filtred_dict(data, edge_types):
    filter_dict = defaultdict(set)

    for edge_type in edge_types:
        for i in data[edge_type].edge_index.T:
            filter_dict[(edge_type, i[0].item())].add(i[1].item())

    return filter_dict


def prepare_evaluation(model, data, test_data, edge_types):
    all_embeddings = get_embeddigs(model, test_data)
    filter_dict = get_filtred_dict(data, edge_types)

    return all_embeddings, filter_dict


@torch.no_grad()
def test(model, test_data, batch_size, edge_types, all_embeddings, filter_dict, k_list=[1, 5, 10, 50, 100], device='cuda'):
    model.eval()
    result_dict = {}
    for edge_type in edge_types:
        mrr = 0
        hits = {k: 0 for k in k_list}
        total_samples = 0

        loader = DataLoader(test_data[edge_type].edge_label_index.T, batch_size=batch_size, shuffle=False)
        for batch in tqdm(loader):
            batch = batch.to(device)
            corr_batch_size = batch.size(0)
            total_samples += corr_batch_size

            heads_index = batch[:, 0]
            targets_index = batch[:, 1]

            all_embeddings[edge_type[0]] = all_embeddings[edge_type[0]].to(device)
            all_embeddings[edge_type[-1]] = all_embeddings[edge_type[-1]].to(device)

            heads = all_embeddings[edge_type[0]][heads_index]

            all_scores = torch.matmul(heads, all_embeddings[edge_type[-1]].T)

            #фильтруем батчами
            mask_b = []
            mask_idx = []
            target_scores = torch.zeros(corr_batch_size, device=device)

            for i in range(corr_batch_size):
                h_eval_id = heads_index[i].item()
                target_t_eval_id = targets_index[i].item()

                #сохраняем целевой скор
                target_scores[i] = all_scores[i, target_t_eval_id]

                #создаем маску истинных триплетов
                true_tails = filter_dict[(edge_type, h_eval_id)]
                for true_t in true_tails:
                    if true_t != target_t_eval_id:
                        mask_b.append(i)
                        mask_idx.append(true_t)

            if mask_b:
                all_scores[mask_b, mask_idx] = -1e9

            ranks = (all_scores > target_scores.unsqueeze(1)).sum(dim=1) + 1

            mrr += (1.0 / ranks).sum().item()
            for k in k_list:
                hits[k] += (ranks <= k).sum().item()

            del all_scores
        print(f"Закончилась оценка для {edge_type}")

        mrr = mrr / total_samples
        hits = {k: v / total_samples for k, v in hits.items()}
        formatted_hits = {k: f"{v:.4f}" for k, v in hits.items()}

        result_dict[edge_type] = (mrr, formatted_hits)



    return result_dict




In [24]:
all_embeddings, filter_dict = prepare_evaluation(model, data=data, test_data=test_data, edge_types=iw_edge_types)
test(model, test_data, batch_size=300, edge_types=iw_edge_types, all_embeddings=all_embeddings, filter_dict=filter_dict)

100%|██████████| 2/2 [00:00<00:00, 45.11it/s]


Закончилась оценка для ('movie', 'to', 'director')


100%|██████████| 5/5 [00:00<00:00, 105.80it/s]


Закончилась оценка для ('movie', 'to', 'actor')


100%|██████████| 2/2 [00:00<00:00, 101.61it/s]


Закончилась оценка для ('director', 'to', 'movie')


100%|██████████| 5/5 [00:00<00:00, 112.96it/s]

Закончилась оценка для ('actor', 'to', 'movie')


{('movie', 'to', 'director'): (0.01616019666613684,
  {1: '0.0047', 5: '0.0211', 10: '0.0304', 50: '0.1030', 100: '0.1897'}),
 ('movie', 'to', 'actor'): (0.013709567545356691,
  {1: '0.0031', 5: '0.0164', 10: '0.0265', 50: '0.1061', 100: '0.1615'}),
 ('director', 'to', 'movie'): (0.0003723297137287238,
  {1: '0.0000', 5: '0.0000', 10: '0.0000', 50: '0.0000', 100: '0.0000'}),
 ('actor', 'to', 'movie'): (0.0013864972569083275,
  {1: '0.0000', 5: '0.0008', 10: '0.0016', 50: '0.0109', 100: '0.0179'})}

In [31]:
@torch.no_grad()
def short_test(data):
    for edge_type in data.edge_types:
        model.eval()
        pred = model(data.x_dict, data.edge_index_dict,
                     data[edge_type].edge_label_index, edge_type=edge_type)
        pred = pred.clamp(min=0, max=5)
        target = data[edge_type].edge_label.float()
        rmse = F.mse_loss(pred, target).sqrt()
    return float(rmse)

In [32]:
short_test(test_data)

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_addmm)